In [1]:
import pandas as pd
df = pd.read_csv('eia_data.csv')

# Filter to only years 1981 and later
df = df[df['Year'] >= 1981]
df = df[df['State'] != 'US']  

# Add Region
region_mapping = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South',
    'SC': 'South', 'VA': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South',
    'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South', 
    'TX': 'South', 'DC': 'South',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest',
    'ND': 'Midwest', 'SD': 'Midwest',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West',
    'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West',
    'HI': 'West', 'OR': 'West', 'WA': 'West'
}

df['Region'] = df['State'].map(region_mapping)

# Add Dominant Energy Source
def get_dominant_energy(row):
    sources = {
        'Coal': row['Coal_Consumption'],
        'Natural Gas': row['NatGas_Consumption'],
        'Nuclear': row['Nuclear_Consumption'],
        'Renewables': row['Renewable_Consumption']
    }
    return max(sources, key=sources.get)

df['Dominant_Energy_Source'] = df.apply(get_dominant_energy, axis=1)

len(df)

2193

In [2]:
import altair as alt
import pandas as pd

# data processing
df_bar = pd.read_csv('eia_data.csv')
df_bar = df_bar[(df_bar['Year'] >= 1981) & (df_bar['State'] != 'US')]
df_bar['Region'] = df_bar['State'].map(region_mapping)
df_bar = df_bar.dropna(subset=['Region'])

# Average CO2 per state across all years
avg_by_state = (
    df_bar.groupby(['State', 'Region'])
    .agg(
        Avg_CO2=('CO2_Emissions', 'mean'),
        Avg_Energy=('Total_Energy_Consumption', 'mean'),
        Years=('Year', lambda x: f"{int(x.min())}\u2013{int(x.max())}")
    )
    .reset_index()
)

# region filter
region_select = alt.selection_point(
    fields=['Region'],
    bind='legend',
    name='region_filter'
)

# color scale
color_scale = alt.Scale(
    domain=['Northeast', 'South', 'Midwest', 'West'],
    range=['#3b6ea5', '#c0552a', '#4a9a6e', '#8b5ea5']
)

# chart
bar = (
    alt.Chart(avg_by_state)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        x=alt.X(
            'State:N',
            sort=alt.EncodingSortField(field='Avg_CO2', order='descending'),
            title='State',
            axis=alt.Axis(labelAngle=-45, labelFontSize=10)
        ),
        y=alt.Y(
            'Avg_CO2:Q',
            title='Avg CO\u2082 Emissions (Million Metric Tons)',
            axis=alt.Axis(grid=True)
        ),
        color=alt.Color(
            'Region:N',
            scale=color_scale,
            legend=alt.Legend(
                title='Region (click to filter)',
                orient='top-right',
                symbolType='circle'
            )
        ),
        opacity=alt.condition(region_select, alt.value(0.85), alt.value(0.1)),
        tooltip=[
            alt.Tooltip('State:N',      title='State'),
            alt.Tooltip('Region:N',     title='Region'),
            alt.Tooltip('Avg_CO2:Q',    title='Avg CO\u2082 (Mil. Mt)',    format='.1f'),
            alt.Tooltip('Avg_Energy:Q', title='Avg Energy (B. Btu)', format=',.0f'),
            alt.Tooltip('Years:N',      title='Year range'),
        ]
    )
    .add_params(region_select)
    .properties(
        width=680,
        height=380,
        title=alt.TitleParams(
            text='Average CO\u2082 Emissions by State (1981\u20132023)',
            subtitle='Sorted by emissions \u00b7 Click a region in the legend to filter \u00b7 Hover for details',
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor='#6b6560'
        )
    )
    .configure_axis(
        labelFont='sans-serif',
        titleFont='sans-serif',
        titleFontSize=12,
        gridColor='#e8e4df'
    )
    .configure_view(strokeWidth=0)
)

bar


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [3]:
import altair as alt
import pandas as pd

# Load and process data for year-by-year energy mix
df_stack = pd.read_csv('eia_data.csv')
df_stack = df_stack[(df_stack['Year'] >= 1981) & (df_stack['State'] != 'US')]
df_stack['Region'] = df_stack['State'].map(region_mapping)
df_stack = df_stack.dropna(subset=['Region'])

# Sum consumption by region and year
region_year = (
    df_stack.groupby(['Region', 'Year'])
    .agg(
        Coal=('Coal_Consumption', 'sum'),
        NatGas=('NatGas_Consumption', 'sum'),
        Nuclear=('Nuclear_Consumption', 'sum'),
        Renewables=('Renewable_Consumption', 'sum'),
    )
    .reset_index()
)

# Compute share (%) for each source
region_year['Total'] = region_year[['Coal','NatGas','Nuclear','Renewables']].sum(axis=1)
for src, col in [('Coal','Coal'),('Natural Gas','NatGas'),('Nuclear','Nuclear'),('Renewables','Renewables')]:
    region_year[src + '_Share'] = (region_year[col] / region_year['Total'] * 100).round(2)

# Melt to long format
share_cols = ['Coal_Share', 'Natural Gas_Share', 'Nuclear_Share', 'Renewables_Share']
melted = region_year.melt(
    id_vars=['Region', 'Year'],
    value_vars=share_cols,
    var_name='Source',
    value_name='Share'
)
melted['Source'] = melted['Source'].str.replace('_Share', '', regex=False)

# Also keep absolute values for tooltip
abs_cols = ['Coal', 'Natural Gas', 'Nuclear', 'Renewables']
melted_abs = region_year.melt(
    id_vars=['Region', 'Year'],
    value_vars=['Coal', 'NatGas', 'Nuclear', 'Renewables'],
    var_name='Source_raw',
    value_name='Abs'
)
src_map = {'Coal': 'Coal', 'NatGas': 'Natural Gas', 'Nuclear': 'Nuclear', 'Renewables': 'Renewables'}
melted_abs['Source'] = melted_abs['Source_raw'].map(src_map)
melted_abs = melted_abs[['Region', 'Year', 'Source', 'Abs']]

df_final = melted.merge(melted_abs, on=['Region', 'Year', 'Source'])

# Year slider
year_slider = alt.binding_range(min=1981, max=2023, step=1, name='Year:  ')
year_select = alt.param(name='year_choice', value=2023, bind=year_slider)

color_scale = alt.Scale(
    domain=['Coal', 'Natural Gas', 'Nuclear', 'Renewables'],
    range=['#0d3b24', '#1a5c38', '#4a9a6e', '#82c9a0']
)

order = ['Coal', 'Natural Gas', 'Nuclear', 'Renewables']

chart = (
    alt.Chart(df_final)
    .transform_filter('datum.Year == year_choice')
    .mark_bar()
    .encode(
        x=alt.X('Region:N', title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('Share:Q',
                title='Share of energy mix (%)',
                stack='zero',
                scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('Source:N',
                        scale=color_scale,
                        sort=order,
                        legend=alt.Legend(title='Source', orient='top')),
        order=alt.Order('Source:N', sort='ascending'),
        tooltip=[
            alt.Tooltip('Region:N',  title='Region'),
            alt.Tooltip('Source:N',  title='Source'),
            alt.Tooltip('Year:O',    title='Year'),
            alt.Tooltip('Share:Q',   title='Share (%)',       format='.1f'),
            alt.Tooltip('Abs:Q',     title='Consumption (bil. Btu)', format=',.0f'),
        ]
    )
    .add_params(year_select)
    .properties(
        width=500,
        height=340,
        title=alt.TitleParams(
            text='Energy Mix by Region',
            subtitle='Use the slider to see how the energy mix changed from 1981 to 2023 · Hover for details',
            fontSize=15,
            subtitleFontSize=11,
            subtitleColor='#6b6560'
        )
    )
    .configure_axis(
        labelFont='sans-serif',
        titleFont='sans-serif',
        titleFontSize=12,
        gridColor='#e8e4df'
    )
    .configure_view(strokeWidth=0)
)

chart


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [4]:
bar.save("summary.html")
chart.save("stacked_bar.html")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c